In [1]:
from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.tag import pos_tag
from nltk.classify import NaiveBayesClassifier, accuracy
from nltk.probability import FreqDist
import os
import pickle
import pandas as pd

MODEL_PATH = './model.pkl'
DATA_PATH = './financial_dataset.csv'

classifier = None
statement = ''

In [2]:
def get_tag(tag):
  if tag.startswith('J'):
    return 'a'
  elif tag.startswith('R'):
    return 'r'
  elif tag.startswith('V'):
    return 'v'
  else:
    return 'n'
  
def lemmatize(tokens):
  tagged = pos_tag(tokens)
  return [WordNetLemmatizer().lemmatize(word, get_tag(tag)) for word, tag in tagged]

def preprocess(sentence):
  sentence = sentence.lower()
  tokens = word_tokenize(sentence)
  tokens = [token for token in tokens if token not in stopwords.words('english')]
  tokens = [token for token in tokens if token.isalpha()]
  tokens = lemmatize(tokens)
  return tokens

In [3]:
df = pd.read_csv(DATA_PATH)

X = df['Statement']
y = df['Sentiment']

all_statement = ' '.join(X)
all_tokens = preprocess(all_statement)
word_frequency = FreqDist(all_tokens)

In [4]:
def extract_features(sentence):
  features = {}
  for word in word_frequency.keys():
    features[word] = word in sentence
  return features

features = [(extract_features(preprocess(sentiment)), statement) for sentiment, statement in zip(X, y)]
features

[({'geosolutions': True,
   'technology': True,
   'leverage': True,
   'benefon': True,
   'gps': True,
   'solution': True,
   'provide': True,
   'location': True,
   'base': True,
   'search': True,
   'community': True,
   'platform': True,
   'relevant': True,
   'multimedia': True,
   'content': True,
   'new': True,
   'powerful': True,
   'commercial': True,
   'model': True,
   'esi': False,
   'low': False,
   'bk': False,
   'real': False,
   'possibility': False,
   'last': False,
   'quarter': False,
   'componenta': False,
   'net': False,
   'sale': False,
   'double': False,
   'period': False,
   'year': False,
   'earlier': False,
   'move': False,
   'zero': False,
   'profit': False,
   'loss': False,
   'spy': False,
   'would': False,
   'surprise': False,
   'see': False,
   'green': False,
   'close': False,
   'shell': False,
   'billion': False,
   'bg': False,
   'deal': False,
   'meet': False,
   'shareholder': False,
   'skepticism': False,
   'ssh': Fals

In [5]:
split = int(len(features) * 0.8)
train_set = features[:split]
test_set = features[split:]

In [6]:
def train_model():
  classifier = NaiveBayesClassifier.train(train_set)
  model_accuracy = accuracy(classifier, test_set)

  classifier.show_most_informative_features(5)
  print("Model accuracy:", model_accuracy, '%')

  with open(MODEL_PATH, 'wb') as f:
    pickle.dump(classifier, f)
  print("Model exported to", MODEL_PATH)

In [7]:
def show_postag(sentence):
  tokens = preprocess(sentence)
  tagged = pos_tag(tokens)

  for word, tag in tagged:
    print(f"{word} - {tag}")

  return tokens

def show_synonym_antonym(tokens):
  for token in tokens:
    synonym = []
    antonym = []

    for syn in wordnet.synsets(token):
      for lemma in syn.lemmas():
        synonym.append(lemma.name())
        for ant in lemma.antonyms():
          antonym.append(ant.name())
    
    print("Word:", token)
    if synonym:
      print('Synonym:', synonym)
    else:
      print('No Synonym')
    if antonym:
      print('Antonym:', antonym)
    else:
      print('No Antonym')

def show_prediction(tokens):
  prediction = classifier.classify(extract_features(tokens))
  print(f'Prediction Sentiment: {prediction}')

def analyze_statement():
  if len(statement) == 0:
    print('Please insert valid statement!')
    return
  else:
    tokens = show_postag(statement)
    show_synonym_antonym(tokens)
    show_prediction(tokens)

In [8]:
while True:
  print('1. Write your statement')
  print('2. Analyze your statement')
  print('3. Exit')

  try:
    user_choice = int(input('>> '))
  except ValueError:
    print('Please insert int only.')
    continue

  if user_choice == 1:
    statement = input('Write your statement: ')
    if statement.index(' ') == -1:
      print('Statement invalid! Statement must be more than 1 word')
    else:
      if os.path.exists(MODEL_PATH):
        with open(MODEL_PATH, 'rb') as f:
          classifier = pickle.load(f)
      else:
        train_model()
  elif user_choice == 2:
    analyze_statement()
  elif user_choice == 3:
    print('Exiting...')
    break
  else:
    print('Please insert valid option! Number must be between 1 and 3')


1. Write your statement
2. Analyze your statement
3. Exit
Please insert int only.
1. Write your statement
2. Analyze your statement
3. Exit
Exiting...
